In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
# ── 1. INSTALL DEPENDENCIES ──
!pip install -q --upgrade transformers>=5.3
!pip install -q omnivoice soundfile gradio==6.11.0

import os
import logging
import torch
import gc
import time
import numpy as np
import soundfile as sf
import gradio as gr
from omnivoice import OmniVoice, OmniVoiceGenerationConfig

# ── 2. SYSTEM ENVIRONMENT & LOGGING SETUP ──
logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(name)s %(levelname)s: %(message)s')
logging.getLogger('omnivoice').setLevel(logging.INFO)
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

# ── 3. INITIALIZE MODEL ON A SINGLE T4 GPU ──
CHECKPOINT = 'k2-fsa/OmniVoice'
print(f"🚀 Loading OmniVoice from {CHECKPOINT} using device_map='cuda:0'...")

model = OmniVoice.from_pretrained(
    CHECKPOINT,
    device_map="cuda:0",  
    dtype=torch.float16,  #[cite: 2]
    load_asr=True,        #[cite: 2]
    token=False,
)
sampling_rate = model.sampling_rate
print(f"✅ Model successfully loaded on cuda:0. Base rate: {sampling_rate} Hz")

# ── 4. GRADIO APP CONFIGURATIONS & DICTIONARIES ──
LANGUAGES = [
    "Auto", "English (en)", "Chinese (zh)", "Japanese (ja)", "Korean (ko)",
    "French (fr)", "German (de)", "Spanish (es)", "Portuguese (pt)",
    "Russian (ru)", "Arabic (ar)", "Hindi (hi)", "Italian (it)",
    "Dutch (nl)", "Turkish (tr)", "Polish (pl)", "Swedish (sv)",
    "Thai (th)", "Vietnamese (vi)", "Indonesian (id)", "Malay (ms)",
]

CATEGORIES = {
    "Gender": ["male", "female"],
    "Age": ["child", "teenager", "young adult", "middle-aged", "elderly"],
    "Pitch": ["very low pitch", "low pitch", "moderate pitch", "high pitch", "very high pitch"],
    "Style": ["whisper"],
    "English Accent": ["american accent", "british accent", "australian accent",
                       "canadian accent", "indian accent", "chinese accent",
                       "japanese accent", "korean accent", "portuguese accent",
                       "russian accent"],
}

# ── 5. CORE GENERATION ENGINE ──
def build_instruct(groups):
    selected = [g for g in groups if g and g != "Auto"]
    return ", ".join(selected) if selected else ""

def generate_speech(text, language, ref_audio, instruct, num_step, guidance_scale, denoise, speed, duration, preprocess_prompt, postprocess_output, mode="clone", ref_text=None):
    if not text or not text.strip():
        return None, "⚠️ Please type some text to synthesize first."

    lang_code = None
    if language and language != "Auto":
        lang_code = language.split("(")[-1].rstrip(")").strip() if "(" in language else language

    gen_config = OmniVoiceGenerationConfig(
        num_step=int(num_step or 32),
        guidance_scale=float(guidance_scale) if guidance_scale is not None else 2.0,
        denoise=bool(denoise) if denoise is not None else True,
        preprocess_prompt=bool(preprocess_prompt),
        postprocess_output=bool(postprocess_output),
    )

    kw = dict(text=text.strip(), language=lang_code, generation_config=gen_config)

    if speed is not None and float(speed) != 1.0:
        kw["speed"] = float(speed)
    if duration is not None and float(duration) > 0:
        kw["duration"] = float(duration)

    if mode == "clone":
        if ref_audio is None:
            return None, "⚠️ Missing reference audio file! Please upload or record a clip."
        kw["voice_clone_prompt"] = model.create_voice_clone_prompt(ref_audio=ref_audio, ref_text=ref_text)
    elif mode == "design":
        if instruct and instruct.strip():
            kw["instruct"] = instruct.strip()

    try:
        audio = model.generate(**kw)
    except Exception as e:
        return None, f"❌ Engine Generation Error: {type(e).__name__}: {e}"
    finally:
        # Prevent VRAM fragmentation leaks between generations
        torch.cuda.empty_cache()
        gc.collect()

    waveform = audio[0].squeeze()
    if hasattr(waveform, 'numpy'):
        waveform = waveform.numpy()
    waveform = (waveform * 32767).astype(np.int16)
    
    # Use timestamps to avoid filename locking bugs
    unique_id = int(time.time())
    output_filename = f"{mode}_output_{unique_id}.wav"
    sf.write(output_filename, waveform, sampling_rate)
    
    return (sampling_rate, waveform), f"✅ Speech generated smoothly! Saved as {output_filename}"

# ── 6. ADVANCED CONFIGURATION ELEMENT ──
def render_advanced_settings():
    with gr.Accordion("⚙️ Advanced Hyperparameters", open=False):
        ns = gr.Slider(8, 64, value=32, step=1, label="Inference Steps (Lower = Faster)")
        gs = gr.Slider(0.0, 10.0, value=3.0, step=0.1, label="Guidance Scale")
        dn = gr.Slider(0.0, 1.0, value=0.8, step=0.05, label="Denoise Ratio")
        sp = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label="Playback Speed Factor")
        du = gr.Slider(0, 30, value=0, step=0.5, label="Forced Duration (0 = Auto)")
        pp = gr.Checkbox(value=True, label="Preprocess Prompt Text")
        po = gr.Checkbox(value=True, label="Postprocess Audio Output")
    return ns, gs, dn, sp, du, pp, po

# ── 7. INTERFACE ASSEMBLY (GRADIO BLOCKS) ──
with gr.Blocks(title="OmniVoice AI App") as demo:
    gr.HTML("<div style='text-align: center;'><h1>🎙️ OmniVoice Zero-Shot TTS</h1><p>Running Optimized on Single T4 GPU Architecture</p></div>")
    
    with gr.Tabs():
        # TAB 1: VOICE CLONING
        with gr.TabItem("🎤 Voice Clone Studio"):
            with gr.Row():
                with gr.Column(scale=1):
                    vc_text = gr.Textbox(label="Text to Synthesize", lines=4, placeholder="What should the cloned voice say?")
                    vc_ref_audio = gr.Audio(label="Reference Voice File (3 to 10 seconds)", type="filepath")
                    vc_ref_text = gr.Textbox(label="Reference Voice Transcript (Optional)", placeholder="Helps accuracy if uploaded voice matches text...")
                    vc_lang = gr.Dropdown(label="Output Language Alignment", choices=LANGUAGES, value="Auto")
                    vc_ns, vc_gs, vc_dn, vc_sp, vc_du, vc_pp, vc_po = render_advanced_settings()
                    vc_btn = gr.Button("🔥 Run Voice Clone", variant="primary")
                with gr.Column(scale=1):
                    vc_audio = gr.Audio(label="Generated Audio Output", type="numpy")
                    vc_status = gr.Textbox(label="Execution Logs / Status")

            def run_clone(text, lang, ref_aud, ref_text, ns, gs, dn, sp, du, pp, po):
                return generate_speech(text, lang, ref_aud, None, ns, gs, dn, sp, du, pp, po, mode="clone", ref_text=ref_text or None)

            vc_btn.click(run_clone, inputs=[vc_text, vc_lang, vc_ref_audio, vc_ref_text, vc_ns, vc_gs, vc_dn, vc_sp, vc_du, vc_pp, vc_po], outputs=[vc_audio, vc_status])

        # TAB 2: VOICE DESIGN
        with gr.TabItem("🎨 Synthetic Voice Designer"):
            with gr.Row():
                with gr.Column(scale=1):
                    vd_text = gr.Textbox(label="Text to Synthesize", lines=4, placeholder="Type the text you want read aloud...")
                    vd_lang = gr.Dropdown(label="Output Language Alignment", choices=LANGUAGES, value="Auto")
                    vd_groups = [gr.Dropdown(label=cat, choices=["Auto"] + choices, value="Auto") for cat, choices in CATEGORIES.items()]
                    vd_ns, vd_gs, vd_dn, vd_sp, vd_du, vd_pp, vd_po = render_advanced_settings()
                    vd_btn = gr.Button("✨ Design & Synthesize Voice", variant="primary")
                with gr.Column(scale=1):
                    vd_audio = gr.Audio(label="Generated Audio Output", type="numpy")
                    vd_status = gr.Textbox(label="Execution Logs / Status")

            def run_design(text, lang, ns, gs, dn, sp, du, pp, po, *groups):
                return generate_speech(text, lang, None, build_instruct(groups), ns, gs, dn, sp, du, pp, po, mode="design")

            vd_btn.click(run_design, inputs=[vd_text, vd_lang, vd_ns, vd_gs, vd_dn, vd_sp, vd_du, vd_pp, vd_po] + vd_groups, outputs=[vd_audio, vd_status])

# Launch configuration with public share link
demo.queue().launch(share=True)

🚀 Loading OmniVoice from k2-fsa/OmniVoice using device_map='cuda:0'...


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

✅ Model successfully loaded on cuda:0. Base rate: 24000 Hz
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://c0fae6a455a4467136.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see re